# Chapter 07: Dense Retrieval without Re-teaching Embeddings

Cosine-style scoring on deterministic vectors; no model required by default.

In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


## Build a tiny deterministic dense index

In [ ]:
from semantic_search import DenseIndex, normalise
import math

records = [
    {"passage_id": "p-r17", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
    {"passage_id": "p-r71", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
]
index = DenseIndex(records, [[1, 0], [0.9, 0.1]], "toy-model")
query = {"values": [1, 0], "model_fingerprint": "toy-model"}
hits = index.rank(query, 2)
for h in hits:
    print(h["passage_id"], round(h["score"], 6))
assert hits[0]["passage_id"] == "p-r17"
assert hits[0]["score"] > hits[1]["score"]

## Identifier confusion: R17 versus R71

In [ ]:
records2 = [
    {"passage_id": "r17-instr", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
    {"passage_id": "r71-instr", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
]
index2 = DenseIndex(records2, [[0.95, 0.05], [0.90, 0.10]], "toy-model")
q2 = {"values": [1, 0], "model_fingerprint": "toy-model"}
for h in index2.rank(q2, 2):
    print(h["passage_id"], round(h["score"], 6))
assert index2.rank(q2, 1)[0]["passage_id"] == 'r17-instr' 

## Topic confusion: topical neighbour is not an instruction

In [ ]:
records3 = [
    {"passage_id": "cache-overview", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
    {"passage_id": "cache-command", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
]
index3 = DenseIndex(records3, [[0.98, 0.02], [0.70, 0.30]], "toy-model")
q3 = {"values": [0.99, 0.01], "model_fingerprint": "toy-model"}
print([(h["passage_id"], round(h["score"], 4)) for h in index3.rank(q3, 2)])
assert index3.rank(q3, 1)[0]["passage_id"] == 'cache-overview' 

## Truncation analogue: tail instruction is absent from the vector

In [ ]:
full = [1.0, 0.0]
truncated_tail = [0.2, 0.8]
recs = [
    {"passage_id": "long-section", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
    {"passage_id": "short-block", "source_id": "s", "document_id": "d", "version_id": "v1", "representation_id": "r", "segmentation_id": "seg", "content_sha256": "h", "scope": "public-synthetic", "current_eligible": True},
]
idx = DenseIndex(recs, [truncated_tail, [0.9, 0.1]], "toy-model")
q = {"values": full, "model_fingerprint": "toy-model"}
print([(h["passage_id"], round(h["score"], 4)) for h in idx.rank(q, 2)])
assert idx.rank(q, 1)[0]["passage_id"] == 'short-block' 

## Optional pinned MiniLM (only if cached)

In [ ]:
import os
from pathlib import Path
try:
    import sentence_transformers
    print("sentence_transformers available:", sentence_transformers.__version__)
except Exception as exc:
    print("optional model not available:", type(exc).__name__)
print("default mode remains deterministic; no weights loaded")

## Try it yourself
Change vectors or queries and observe ranking changes.

In [ ]:
for qv in [[1,0],[0,1],[0.5,0.5]]:
    qq = {"values": qv, "model_fingerprint": "toy-model"}
    print(qv, "->", [h["passage_id"] for h in index.rank(qq, 1)])

## Observed versus not established
Observed: cosine ordering on declared toy vectors. Not established: MiniLM behaviour, relevance, admission, use or helpfulness.